In [ ]:
# Convert demographics to wide household x demographic format
ET_demo = ET_d.unstack('k').fillna(0) if isinstance(ET_d, pd.Series) else ET_d.copy()
SE_demo = SE_d.unstack('k').fillna(0) if isinstance(SE_d, pd.Series) else SE_d.copy()

# Keep only demographic groups appearing in RDI table
common_ET = ET_demo.columns.intersection(rdi.columns)
common_SE = SE_demo.columns.intersection(rdi.columns)

ET_demo = ET_demo[common_ET]
SE_demo = SE_demo[common_SE]

ET_rdi = rdi[common_ET]
SE_rdi = rdi[common_SE]

# Household nutrient requirements
ET_req = ET_rdi @ ET_demo.T
SE_req = SE_rdi @ SE_demo.T

# Align matrices
ET_N_aligned, ET_req = ET_N.align(ET_req, join='inner', axis=0)
ET_N_aligned, ET_req = ET_N_aligned.align(ET_req, join='inner', axis=1)

SE_N_aligned, SE_req = SE_N.align(SE_req, join='inner', axis=0)
SE_N_aligned, SE_req = SE_N_aligned.align(SE_req, join='inner', axis=1)

# Adequacy ratios
ET_adequacy = ET_N_aligned / ET_req
SE_adequacy = SE_N_aligned / SE_req

# Deficiency indicators
ET_deficient = ET_adequacy < 1
SE_deficient = SE_adequacy < 1

# Share deficient
ET_deficiency_rates = ET_deficient.mean(axis=1) * 100
SE_deficiency_rates = SE_deficient.mean(axis=1) * 100

# Comparative table
comparative_nutrition = pd.DataFrame({
    "Ethiopia % Deficient": ET_deficiency_rates,
    "Senegal % Deficient": SE_deficiency_rates
})

comparative_nutrition["Larger Challenge In"] = comparative_nutrition.apply(
    lambda row: "Ethiopia"
    if row["Ethiopia % Deficient"] > row["Senegal % Deficient"]
    else "Senegal"
    if row["Senegal % Deficient"] > row["Ethiopia % Deficient"]
    else "Similar",
    axis=1
)

comparative_nutrition = comparative_nutrition.sort_values(
    by=["Ethiopia % Deficient", "Senegal % Deficient"],
    ascending=False
)

display(comparative_nutrition)

# Key nutrients
display(
    comparative_nutrition.loc[
        comparative_nutrition.index.intersection(["Energy", "Protein", "Vitamin A"])
    ]
)

# Policy goal suggestions
ET_biggest = comparative_nutrition["Ethiopia % Deficient"].idxmax()
SE_biggest = comparative_nutrition["Senegal % Deficient"].idxmax()

print("\nEthiopia Policy Goal:")
print(
    f"Reduce {ET_biggest} deficiency from "
    f"{comparative_nutrition.loc[ET_biggest, 'Ethiopia % Deficient']:.1f}% "
    f"to {comparative_nutrition.loc[ET_biggest, 'Ethiopia % Deficient']/2:.1f}%."
)

print("\nSenegal Policy Goal:")
print(
    f"Reduce {SE_biggest} deficiency from "
    f"{comparative_nutrition.loc[SE_biggest, 'Senegal % Deficient']:.1f}% "
    f"to {comparative_nutrition.loc[SE_biggest, 'Senegal % Deficient']/2:.1f}%."
)